This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

# Decision Tree Regressor from scratch

Here I implement a decision tree algorithm for regression from scratch (no scikit-learn). I build it up from small pieces — the splitting criterion, the split search, and the node structure — and then assemble everything into a single `DecisionTreeRegressor` class.

## Step 1 — MSE and weighted MSE

The impurity criterion. `mse` measures the spread of target values in a node; `weighted_mse` combines the MSE of the two child nodes weighted by their sizes — this is what a candidate split is scored by.

In [ ]:
import numpy as np

def mse(y: np.ndarray) -> float:
    """Compute the mean squared error of a vector."""
    return np.mean((y - np.mean(y))**2)


def weighted_mse(y_left: np.ndarray, y_right: np.ndarray) -> float:
    """Compute the weighted mean squared error of two vectors."""
    w_mse = (len(y_left) * mse(y_left) + len(y_right) * mse(y_right)) \
            / (len(y_left) + len(y_right))
    return w_mse

## Step 2 — best split on a single feature

`split` takes the feature matrix, the target, and a feature index, and finds the threshold on that feature that minimizes the weighted MSE.

**Approach:** I walk along the sorted values of the chosen feature. When there is a run of equal values (e.g. `[..., 1, 1, 1, 3]`), I place the split after the last of them, so equal values never end up on different sides.

In [ ]:
import numpy as np


def split(X: np.ndarray, y: np.ndarray, feature: int) -> tuple[np.float64, np.float64]:
    """Find the best split for a node (one feature)"""
    
    if len(y) < 2:
        return None

    idx = np.argsort(X[:, feature])
    x = np.sort(X[:, feature])
    y = y[idx]
    
    best_split = None
    best_mse = mse(y)
    
    for i in range(1, len(x)):
        if x[i-1] != x[i]:
            y_left = y[0:i]
            y_right = y[i:]
            split_mse = weighted_mse(y_left, y_right)
            if split_mse < best_mse:
                best_mse = split_mse
                best_split = i - 1
    
    return best_mse, x[best_split] if best_split is not None else None

## Step 3 — best split across all features

`best_split` scans every feature, reuses `split` on each, and returns the `(feature, threshold)` pair with the lowest weighted MSE.

In [ ]:
from __future__ import annotations

def best_split(X: np.ndarray, y: np.ndarray) -> tuple[int, float]:
    """Find the best split for a node among all features"""
    s = X.shape[1]
    
    best_feature = None
    best_threshold = None
    best_mse = mse(y)
    
    for i in range(s):
        current_mse, threshold = split(X, y, i)
        if current_mse < best_mse:
            best_feature = i
            best_threshold = threshold
            best_mse = current_mse
    
    return best_feature, best_threshold

## Step 4 — the Node

A `Node` dataclass holds the information for a tree node or leaf:
- `feature` — index of the feature used for the split
- `threshold` — split threshold value
- `n_samples` — number of samples in the node
- `value` — rounded mean of the target (the leaf prediction)
- `mse` — MSE criterion at the node
- `left`, `right` — child nodes

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

@dataclass
class Node:
    """Decision tree node
    Parameters
    ----------
    feature : int, optional (default=None)
        The feature index used for splitting the node.
    threshold : float, optional (default=None)
        The threshold value at the node.
    n_samples : int, optional (default=None)
        The number of samples at the node.
    value : int, optional (default=None)
        The value of the node (i.e., the mean target value of the samples at the node).
    mse : float, optional (default=None)
        The mean squared error of the node (i.e., the impurity criterion).
    left : Node, optional (default=None)
        The left child node.
    right : Node, optional (default=None)
        The right child node.
    """
    
    feature: int = None
    threshold: float = None
    n_samples: int = None
    value: int = None
    mse: float = None
    left: Node = None
    right: Node = None

## Putting it together: `DecisionTreeRegressor`

Now I gather the building blocks above into one class (the helpers become private methods) and add the parts that make it a usable estimator:

- **`fit` / `_split_node`** — recursively builds the tree. At each node it tries to split, and recurses into the left and right children. It stops when any stopping condition is met:
  - the maximum depth `max_depth` is reached;
  - the node has fewer than `min_samples_split` samples;
  - the split does not improve the metric — which happens when the remaining values are identical, or the left and right means coincide.
  The built tree is stored in the `tree_` attribute, whose nodes are `Node` instances.

- **`as_json` / `_as_json`** — serializes the tree into a JSON string, recursively walking from the root down to the leaves.

- **`predict` / `_predict_one_sample`** — for each sample, descends the tree until it reaches a leaf and returns that leaf's `value`; `predict` applies this over a matrix `X` and returns a 1-D array of predictions.

In [ ]:
from __future__ import annotations

import json

import numpy as np
from dataclasses import dataclass


@dataclass
class DecisionTreeRegressor:
    """Decision tree regressor."""
    max_depth: int
    min_samples_split: int = 2

    def fit(self, X: np.ndarray, y: np.ndarray) -> DecisionTreeRegressor:
        """Build a decision tree regressor from the training set (X, y)."""
        self.n_features_ = X.shape[1]
        self.tree_ = self._split_node(X, y)
        return self

    def _mse(self, y: np.ndarray) -> float:
        """Compute the mse criterion for a given set of target values."""
        return np.mean((y - np.mean(y))**2)

    def _weighted_mse(self, y_left: np.ndarray, y_right: np.ndarray) -> float:
        """Compute the weighted mse criterion for a two given sets of target values"""
        w_mse = (len(y_left) * self._mse(y_left) + len(y_right) * self._mse(y_right)) \
                / (len(y_left) + len(y_right))
        return w_mse

    def _split(self, X: np.ndarray, y: np.ndarray, feature: int) -> tuple[np.float64, np.float64]:
        """Find the best split for a one feature"""
        if len(y) < 2:
            return None

        idx = np.argsort(X[:, feature])
        x = np.sort(X[:, feature])
        y = y[idx]

        best_split = None
        best_mse = self._mse(y)

        for i in range(1, len(x)):
            if x[i-1] != x[i]:
                y_left = y[0:i]
                y_right = y[i:]
                split_mse = self._weighted_mse(y_left, y_right)
                if split_mse < best_mse:
                    best_mse = split_mse
                    best_split = i - 1

        return best_mse, x[best_split] if best_split is not None else None

    def _best_split(self, X: np.ndarray, y: np.ndarray) -> tuple[int, float]:
        """Find the best split for a node among all features."""
        s = X.shape[1]

        best_feature = None
        best_threshold = None
        best_mse = self._mse(y)

        for i in range(s):
            current_mse, threshold = self._split(X, y, i)
            if current_mse < best_mse:
                best_feature = i
                best_threshold = threshold
                best_mse = current_mse

        return best_feature, best_threshold

    def _split_node(self, X: np.ndarray, y: np.ndarray, depth: int = 0) -> Node:
        """Split a node and return the resulting left and right child nodes.
            This function works in a recursive way so the funciton is called
            each time when the node can be splitted."""

        n_samples = X.shape[0]
        value = round(np.mean(y))
        mse = self._mse(y)

        node = Node(n_samples=n_samples,
                    value=value,
                    mse=mse)

        #Check if the node can be splitted
        if depth < self.max_depth and n_samples >= self.min_samples_split:

            feature, threshold = self._best_split(X, y)
            
            #threshold=None means that further split doesn't improve metrics
            if threshold is None:
                return node

            #Split X and target array
            mask = X[:, feature] <= threshold
            X_left = X[mask]
            y_left = y[mask]
            X_right = X[~mask]
            y_right = y[~mask]

            depth += 1
            left_node = self._split_node(X_left, y_left, depth)
            right_node = self._split_node(X_right, y_right, depth)

            node.feature = feature
            node.threshold = threshold
            node.left = left_node
            node.right = right_node

        return node
    
    def as_json(self) -> str:
        """Return the decision tree as a JSON string."""
        tree = json.dumps(self._as_json(self.tree_))
        return tree
    
    def _as_json(self, node: Node) -> dict:
        """Build a JSON-serializable dict for the subtree. Execute recursively."""
        if node.left is None:
            leaf = {
                "value": int(node.value),
                "n_samples": int(node.n_samples),
                "mse": round(float(node.mse), 2),
            }
            return leaf
        
        node_dict = {
            "feature": int(node.feature),
            "threshold": int(round(node.threshold)),
            "n_samples": int(node.n_samples),
            "mse": round(float(node.mse), 2),
            "left": self._as_json(node.left),
            "right": self._as_json(node.right)
        }
        
        return node_dict
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict regression target for X.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            The input samples.

        Returns
        -------
        y : array of shape (n_samples,)
            The predicted values.
        """
        predictions = np.array([], dtype=int)
        for sample in X:
            prediction = self._predict_one_sample(sample)
            predictions = np.append(predictions, prediction)

        return predictions


    def _predict_one_sample(self, features: np.ndarray) -> int:
        """Predict the target value of a single sample."""
        node = self.tree_
        while node.left is not None:
            if features[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right    
            
        return node.value
